# 01 — Exploring a SFINCS flood map

**Goal**: Load the maximum flood depth raster produced by the pipeline and compute basic flood metrics.  
**Prerequisite**: A completed SFINCS run (i.e. `sfincs_postprocess.py` has been executed for your event).  
**No simulation is run here** — this notebook only reads existing outputs.

---

## What the pipeline produces

After running the full pipeline, each SFINCS scenario folder contains a `plot_output/` subdirectory with:

| File | Description |
|------|-------------|
| `sfincs_output_hmax_AllTime.tif` | Maximum flood depth [m] over the entire simulation, downscaled to the high-resolution subgrid DEM and masked for permanent water bodies (GSWO). This is the primary result file. |
| `sfincs_output_hmax_period_*.tif` | Daily flood depth maxima (one per day). |
| `sfincs_basemap.png` | Overview map of the model domain. |
| `sfincs_forcing.png` | Summary of the meteorological forcing applied. |

The **hmax_AllTime.tif** is a single-band GeoTIFF in **EPSG:4326** (WGS84) with values in **metres**. Cells with no flooding have value 0 or are masked (NaN).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import geopandas as gpd
import rioxarray as rxr
import contextily as ctx
from pathlib import Path

## Paths

Edit the cells below to point to your event and scenario.  
The scenario folder name encodes the forcing configuration: `CF0` means no counterfactual adjustment (i.e. the **factual** present-day climate run).

In [ ]:
# ── Edit these for your event ────────────────────────────────────────────────
BASE   = Path("/p/11210471-001-compass/03_Runs")
REGION = "sofala"
EVENT  = "Idai"

# Factual scenario: all CF values are 0 (no climate adjustment)
SCENARIO = "event_tp_era5_hourly_zarr_CF0_GTSMv41_CF0_era5_hourly_spw_IBTrACS_CF0"
# ─────────────────────────────────────────────────────────────────────────────

SFINCS_DIR = BASE / REGION / EVENT / "sfincs" / SCENARIO
HMAX_TIF   = SFINCS_DIR / "plot_output" / "sfincs_output_hmax_AllTime.tif"
REGION_GEO = SFINCS_DIR / "gis" / "region.geojson"

# Guard: print a warning if the run hasn't been executed yet
for p in [HMAX_TIF, REGION_GEO]:
    if not p.exists():
        print(f"⚠️  Not found: {p}")
    else:
        print(f"✓  {p.name}")

## The model domain

The `gis/region.geojson` file is a single polygon defining the outer boundary of the SFINCS model grid. It is produced during model setup (`SfincsModel.build()`) and is used here to add context to the flood map.

In [ ]:
region = gpd.read_file(REGION_GEO)

print(f"CRS  : {region.crs}")
print(f"Bounds (xmin, ymin, xmax, ymax):")
print(f"  {region.total_bounds.round(4)}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
region.boundary.plot(ax=ax, color="red", linewidth=1.5, label="Model domain")
ctx.add_basemap(ax, crs=region.crs.to_string(), source=ctx.providers.CartoDB.Positron, zoom=9)
ax.set_title(f"SFINCS model domain — TC {EVENT}, {REGION.title()}", fontsize=11)
ax.legend()
plt.tight_layout()

## The flood depth raster

`rioxarray` reads GeoTIFFs as `xarray.DataArray` objects, preserving the CRS and spatial metadata. `masked=True` replaces the nodata value with `NaN` so arithmetic operations work correctly.

In [ ]:
hmax = rxr.open_rasterio(HMAX_TIF, masked=True).squeeze()  # drop the band dimension

print(f"Shape  : {hmax.shape}  (rows × cols)")
print(f"CRS    : {hmax.rio.crs}")
print(f"Res    : {hmax.rio.resolution()} degrees")
print(f"Values : min={float(hmax.min()):.3f} m,  max={float(hmax.max()):.3f} m")
print(f"Flooded pixels (> 0): {int((hmax > 0).sum())}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))

# Mask dry cells so they are transparent
flooded = hmax.where(hmax > 0)

flooded.plot(
    ax=ax,
    cmap="Blues",
    vmin=0.05,
    vmax=3.0,
    add_colorbar=True,
    cbar_kwargs={"label": "Max flood depth [m]", "shrink": 0.7},
    alpha=0.85,
)
region.boundary.plot(ax=ax, color="black", linewidth=0.8, linestyle="--", label="Model domain")
ctx.add_basemap(ax, crs=hmax.rio.crs.to_string(), source=ctx.providers.CartoDB.Positron, zoom=10)

ax.set_title(f"TC {EVENT} — Maximum flood depth (factual)", fontsize=12)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
plt.tight_layout()

## Flood metrics

Three summary statistics are typically reported for a flood event:

| Metric | Formula | Unit |
|--------|---------|------|
| **Extent** | number of flooded pixels × pixel area | km² |
| **Volume** | sum of (depth × pixel area) | m³ |
| **Mean depth** | mean depth over flooded pixels only | m |

Because the raster is in geographic coordinates (degrees), the pixel area varies with latitude. We use the approximation:  
`pixel_area_m² ≈ (res_deg × 111 320)² × cos(lat_rad)`,  
where 111 320 m ≈ 1 degree of latitude.

In [ ]:
FLOOD_THRESHOLD = 0.05  # metres — ignore pixels shallower than 5 cm

# Pixel dimensions (degrees)
res_x = abs(float(hmax.rio.resolution()[0]))  # lon resolution
res_y = abs(float(hmax.rio.resolution()[1]))  # lat resolution

# Reference latitude for area correction (centre of domain)
lat_ref = float(hmax.y.mean())
lat_rad = np.deg2rad(lat_ref)

# Approximate pixel area [m²] — valid for small domains (< ~5°)
pixel_area_m2 = (res_x * 111_320) * (res_y * 111_320) * np.cos(lat_rad)
print(f"Pixel area: {pixel_area_m2:.1f} m²  ({pixel_area_m2/1e4:.4f} ha)")

# Flooded mask (above threshold)
flooded_mask = hmax.values > FLOOD_THRESHOLD
depths = hmax.values[flooded_mask]

n_flooded   = flooded_mask.sum()
extent_km2  = n_flooded * pixel_area_m2 / 1e6
volume_m3   = depths.sum() * pixel_area_m2
mean_depth  = depths.mean()

print()
print("─" * 40)
print(f"  Flood extent :  {extent_km2:>10.2f} km²")
print(f"  Flood volume :  {volume_m3/1e6:>10.2f} Mm³")
print(f"  Mean depth   :  {mean_depth:>10.3f} m")
print(f"  Max depth    :  {depths.max():>10.3f} m")
print("─" * 40)

---

## Next steps

- **`02_flood_timeseries.ipynb`** — explore how the flood evolved day by day using the raw `sfincs_map.nc` output.
- **`03_climate_attribution.ipynb`** — compare the factual scenario against counterfactual runs to quantify the influence of climate change.
- **`04_damage_analysis.ipynb`** — load FIAT building-level damage outputs and map economic impact.